# Example 0.5 / 1: Solve and Read Results

Companion notebook for **0.5 Running AMPL from Python** in the AMPL lab Documentation, section "Solve and Read Results". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve and Read Results

The following box puts the model, data, solve, and result extraction together:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # Products that can be manufactured
param profit {PROD};                      # Profit per ton
param rate {PROD} > 0;                    # Production rate in tons per hour
param market {PROD} >= 0;                 # Maximum sales in tons
param avail >= 0;                         # Available production hours

var Make {p in PROD} >= 0, <= market[p];  # Tons to manufacture

maximize TotalProfit:                     # Total profit in dollars
    sum {p in PROD} profit[p] * Make[p];

subject to TimeLimit:                     # Hours used cannot exceed available hours
    sum {p in PROD} Make[p] / rate[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]  # Load product members
ampl.param["profit"] = {"bands": 25, "coils": 30}  # Load profit per ton
ampl.param["rate"] = {"bands": 200, "coils": 140}  # Load tons per hour
ampl.param["market"] = {"bands": 6000, "coils": 4000}  # Load market limits
ampl.param["avail"] = 40  # Load available hours

In [5]:
# run.py
ampl.solve(solver="highs")  # Solve the current model and data with HiGHS
assert ampl.solve_result == "solved", ampl.solve_result  # Stop on a bad status
profit = ampl.obj["TotalProfit"].value()  # Read the scalar objective value
plan = ampl.var["Make"].to_pandas()  # Copy indexed variable values to pandas
print(ampl.solve_result)  # Report the solve status
print(profit)  # Report the optimal objective
print(plan)  # Report the production quantities

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
solved
192000.0
       Make.val
bands      6000
coils      1400


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils;

param: profit rate market :=
    bands  25  200  6000
    coils  30  140  4000;

param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display solve_result;
display TotalProfit;
display Make;

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
solve_result = solved

TotalProfit = 192000

Make [*] :=
bands  6000
coils  1400
;

